# Pfennig 2024 Synechocystis model

Reference: Tobias Pfennig, Elena Kullmann, Tomáš Zavřel, Andreas Nakielski, \
Oliver Ebenhöh, Jan Červený, Gábor Bernát, Anna Barbara Matuszyńska  
"Shedding light on blue-green photosynthesis: A wavelength-dependent mathematical model of photosynthesis in Synechocystis sp. PCC 6803"  
PLOS Computational Biology 20.9 (2024): e1012445.  

[doi: 10.1371/journal.pcbi.1012445](https://www.doi.org/10.1371/journal.pcbi.1012445)

In [ ]:
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mxlpy import Model
from mxlpy.parallel import Cache, parallelise

from mxlmodels import Simulator, data, get_pfennig2024_synechocystis, plot

In [ ]:
DEFAULT = data.pfennig2024.default()
LIGHTS = data.pfennig2024.lights()

model = get_pfennig2024_synechocystis(
    light_spectrum=DEFAULT.light_spectrum,
    light_spectrum_measure=DEFAULT.light_spectrum_measure,
    ocp_absorption=DEFAULT.ocp_absorption_per_wavelength,
    abs_coef=DEFAULT.pigment_abs_coef_per_wavelength,
    molar_masses=DEFAULT.molar_masses,
    ps_comp=DEFAULT.ps_comp,
    pigment_content=DEFAULT.pigment_content,
)

res = Simulator(model).simulate(10).get_result().unwrap_or_err()
fig, ax = plot.lines(res.fluxes.loc[:, ["vPS2"]])

## Light sources

In [ ]:
def plot_spectrum(ax, s):
    # Wavelength anchor points and their approximate sRGB colors.
    # Positions are normalized over 400–700 nm so spacing is physically correct.
    _WL_MIN, _WL_MAX = 400, 700
    _SPECTRUM = [
        (400, (0.58, 0.00, 1.00)),  # violet limit
        (450, (0.00, 0.00, 1.00)),  # blue
        (500, (0.00, 1.00, 1.00)),  # cyan
        (550, (0.00, 1.00, 0.00)),  # green
        (580, (1.00, 1.00, 0.00)),  # yellow
        (600, (1.00, 0.50, 0.00)),  # orange
        (650, (1.00, 0.00, 0.00)),  # red
        (700, (1.00, 0.00, 0.00)),  # red limit
    ]

    spectral_cmap = mcolors.LinearSegmentedColormap.from_list(
        "visible_spectrum",
        [((wl - _WL_MIN) / (_WL_MAX - _WL_MIN), color) for wl, color in _SPECTRUM],
    )

    wls = s.index.to_numpy(dtype=float)
    vals = s.to_numpy(dtype=float)

    ax.plot(wls, vals, color="k", linewidth=1)

    # Invisible fill_between to obtain the clip path
    polygon = ax.fill_between(wls, vals, alpha=0)

    # Spectral gradient clipped to the area under the curve
    gradient = ax.imshow(
        np.linspace(0, 1, 256).reshape(1, -1),
        cmap=spectral_cmap,
        aspect="auto",
        extent=[wls.min(), wls.max(), 0, vals.max()],
        alpha=0.4,
    )
    gradient.set_clip_path(polygon.get_paths()[0], transform=ax.transData)
    return ax


fig, axs = plt.subplots(1, 4, figsize=(12, 3), layout="constrained")

plot_spectrum(axs[0], LIGHTS["solar"] / LIGHTS["solar"].max()).set(title="Solar")
plot_spectrum(
    axs[1], LIGHTS["fluorescent_lamp"] / LIGHTS["fluorescent_lamp"].max()
).set(title="Fluorescent lamp")
plot_spectrum(axs[2], LIGHTS["cool_white_led"] / LIGHTS["cool_white_led"].max()).set(
    title="Cool white LEd"
)
plot_spectrum(axs[3], LIGHTS["warm_white_led"] / LIGHTS["warm_white_led"].max()).set(
    title="Warm white LED"
)

for ax in axs:
    ax.set(xlabel="Wavelength (nm)", ylabel="Intensity", ylim=(0, 1.05))

## Re-create fig 

In [ ]:
def _light_variant(light_spectrum: pd.Series) -> Model:
    return get_pfennig2024_synechocystis(
        light_spectrum=light_spectrum,
        light_spectrum_measure=DEFAULT.light_spectrum_measure,
        ocp_absorption=DEFAULT.ocp_absorption_per_wavelength,
        abs_coef=DEFAULT.pigment_abs_coef_per_wavelength,
        molar_masses=DEFAULT.molar_masses,
        ps_comp=DEFAULT.ps_comp,
        pigment_content=DEFAULT.pigment_content,
    )


def _worker(inp: tuple[str, float]) -> float:
    light, pfd = inp
    return (
        Simulator(_light_variant(LIGHTS[light] * pfd / 100))
        .simulate(10_000)
        .get_result()
        .unwrap_or_err()
        .get_fluxes()["vCCM"]
        .iloc[-1]
    )


def _read_scan(carb_cons: list[tuple[str, float]]) -> pd.DataFrame:
    res = {}
    for k, v in carb_cons:
        light, pfd = k.split("-")
        res.setdefault(light, {})[pfd] = v
    return pd.DataFrame(res)


inp = pd.concat(
    pd.DataFrame(
        {
            "light": i,
            "pfd": np.arange(20, 500 + (step := 40), step, dtype=float),
        }
    )
    for i in [
        "solar",
        "fluorescent_lamp",
        "cool_white_led",
        "warm_white_led",
    ]
)


carb_cons = parallelise(
    _worker,
    inputs=list(
        zip(
            [f"{v['light']}-{v['pfd']:.0f}" for _, v in inp.iterrows()],
            [tuple(v) for _, v in inp.iterrows()],
            strict=True,
        )
    ),
    cache=Cache(Path(".cache") / "pfennig2024-carb-cons"),
)


_read_scan(carb_cons).plot()